## unitary/toxic-bert


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "notebooks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'unitary/toxic-bert'
TARGET_MODULES = ['query', 'key', 'value', 'dense']
SCORE_STRATEGY = 'softmax_positive'
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
MAX_LENGTH = 256
BATCH_SIZE = 8
LR = 2e-5
EPOCHS = 1

set_seed(42)
sns.set_theme(style="whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
def load_text_frame() -> pd.DataFrame:
    for candidate in [Path(path) for path in DATASET_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.suffix == ".parquet":
            frame = pd.read_parquet(candidate)
        else:
            frame = pd.read_csv(candidate)
        break
    else:
        raise FileNotFoundError("No local dataset was found for this notebook.")

    rename = {}
    for column in frame.columns:
        low = column.lower()
        if low in {"comment_text", "content", "sentence"}:
            rename[column] = TEXT_COLUMN
        elif low in {"target", "class"}:
            rename[column] = LABEL_COLUMN
    frame = frame.rename(columns=rename)

    if TEXT_COLUMN not in frame.columns:
        raise KeyError("Expected a text column in the dataset.")
    if LABEL_COLUMN not in frame.columns:
        toxic_columns = [col for col in frame.columns if "toxic" in col.lower() or "insult" in col.lower()]
        if toxic_columns:
            frame[LABEL_COLUMN] = frame[toxic_columns].apply(pd.to_numeric, errors="coerce").fillna(0).max(axis=1)
        else:
            frame[LABEL_COLUMN] = 0

    frame[LABEL_COLUMN] = (pd.to_numeric(frame[LABEL_COLUMN], errors="coerce").fillna(0) >= 0.5).astype(int)
    frame[TEXT_COLUMN] = frame[TEXT_COLUMN].astype(str).fillna("")
    frame = frame[[TEXT_COLUMN, LABEL_COLUMN]].dropna().drop_duplicates().reset_index(drop=True)
    return frame


text_df = load_text_frame()
train_df, valid_df = train_test_split(
    text_df,
    test_size=0.2 if len(text_df) >= 50 else 0.3,
    stratify=text_df[LABEL_COLUMN] if text_df[LABEL_COLUMN].nunique() > 1 else None,
    random_state=42,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    }
)


def tokenize_batch(batch):
    return tokenizer(batch[TEXT_COLUMN], truncation=True, max_length=MAX_LENGTH)


encoded = dataset.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }



def scores_from_logits(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits)
    if SCORE_STRATEGY == "sigmoid_any_except_non_toxic":
        proba = 1 / (1 + np.exp(-logits))
        if proba.ndim == 1:
            return proba
        if proba.shape[1] == 1:
            return proba[:, 0]
        non_toxic = proba[:, 0]
        toxic_axes = proba[:, 1:].max(axis=1)
        return 1 - non_toxic * (1 - toxic_axes)
    if logits.ndim == 1 or logits.shape[1] == 1:
        return 1 / (1 + np.exp(-logits.reshape(-1)))
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    probs = exp / exp.sum(axis=1, keepdims=True)
    return probs[:, -1]


inference_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device).eval()

sample_texts = [
    "You are amazing, thank you for helping with this issue.",
    "I hate you and your stupid idea.",
    "This travel post contains a lot of spam links and insults.",
]
perturbations = []
for text in sample_texts:
    perturbations.extend(
        [
            {"variant": "original", "text": text},
            {"variant": "lowercase", "text": text.lower()},
            {"variant": "masked", "text": re.sub(r"[aeiouаеёиоуыэюя]", "*", text, flags=re.IGNORECASE)},
            {"variant": "punctuation_drop", "text": re.sub(r"[!?.,]", "", text)},
        ]
    )


@torch.inference_mode()
def predict_texts(texts: list[str]) -> np.ndarray:
    all_scores = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = tokenizer(texts[start : start + BATCH_SIZE], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        logits = inference_model(**batch).logits.detach().cpu().numpy()
        all_scores.append(scores_from_logits(logits))
    return np.concatenate(all_scores)


inference_df = pd.DataFrame(perturbations)
inference_df["score"] = predict_texts(inference_df["text"].tolist())
inference_df["prediction"] = (inference_df["score"] >= 0.5).astype(int)
display(inference_df)


In [ ]:
inspection_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, output_hidden_states=True, output_attentions=True).to(device).eval()
batch = tokenizer(valid_df[TEXT_COLUMN].head(min(8, len(valid_df))).tolist(), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = inspection_model(**batch)

hidden_norms = torch.stack([layer.norm(dim=-1).mean(dim=1).cpu() for layer in outputs.hidden_states], dim=0).numpy()
attention_summary = torch.stack([layer.mean(dim=(1, 2, 3)).cpu() for layer in outputs.attentions], dim=0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(hidden_norms, cmap="mako", ax=axes[0])
sns.heatmap(attention_summary, cmap="crest", ax=axes[1])
plt.tight_layout()


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }



def trainer_metrics(eval_pred):
    logits, labels = eval_pred
    scores = scores_from_logits(logits)
    return compute_binary_metrics(labels, scores, threshold=0.5)


def build_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        ignore_mismatched_sizes=True,
    )


def run_training(model, run_name: str):
    training_args = TrainingArguments(
        output_dir=str(ARTIFACT_ROOT / run_name),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=10,
        report_to="none",
        load_best_model_at_end=False,
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded["train"],
        eval_dataset=encoded["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=trainer_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model()
    tokenizer.save_pretrained(ARTIFACT_ROOT / run_name)
    return metrics


base_metrics = run_training(build_base_model(), "text_base_" + MODEL_ID.split("/")[-1].replace("-", "_"))
base_metrics


In [ ]:
def build_peft_model(use_dora: bool = False):
    model = build_base_model()
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=TARGET_MODULES,
        modules_to_save=["classifier"],
        use_dora=use_dora,
    )
    peft_model = get_peft_model(model, config)
    peft_model.print_trainable_parameters()
    return peft_model


lora_metrics = run_training(build_peft_model(use_dora=False), "text_lora_" + MODEL_ID.split("/")[-1].replace("-", "_"))
dora_metrics = run_training(build_peft_model(use_dora=True), "text_dora_" + MODEL_ID.split("/")[-1].replace("-", "_"))
pd.DataFrame([{{"run": "base", **base_metrics}}, {{"run": "lora", **lora_metrics}}, {{"run": "dora", **dora_metrics}}])
